In [1]:
# -*- coding: utf-8 -*-


# --- BEGIN TAGS PATCH (must run before sklearn is imported) ---
# --- BEGIN TAGS PATCH (must run before any SearchCV) ---
from sklearn.base import BaseEstimator
if not hasattr(BaseEstimator, "__sklearn_tags__"):
    BaseEstimator.__sklearn_tags__ = lambda self: {}

from scikeras.wrappers import KerasRegressor
if not hasattr(KerasRegressor, "__sklearn_tags__"):
    # 返回一个空 dict 或者根据需要设置标签
    KerasRegressor.__sklearn_tags__ = lambda self: {}
# --- END TAGS PATCH ---

import joblib
import os
# 一定要在 import tensorflow 之前设置
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0=ALL, 1=INFO, 2=WARNING, 3=ERROR
import os
# from openpyxl import load_workbook
from openpyxl import Workbook, load_workbook

EXCEL_PATH = 'model_results.xlsx'

import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    mean_squared_log_error,
    explained_variance_score,
    max_error,
    r2_score
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from skopt import BayesSearchCV
from tensorflow.keras import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, Dense
from tensorflow.keras.layers import GRU
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense
# from tensorflow.keras.wrappers.scikit_learn import KerasRegressor


from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from skopt import BayesSearchCV
import time
import argparse
from skopt.space import Real, Integer, Categorical
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 只保留 ERROR，不显示 INFO、WARNING

# 然后再 import
import warnings
warnings.filterwarnings('ignore')  # 全局忽略所有 warnings

import logging
tf.get_logger().setLevel(logging.ERROR)     # Keras/TF 自己的 logger
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# parser = argparse.ArgumentParser()
# parser.add_argument('--seed', type=int, default=42,
#                     help='random seed')
# args = parser.parse_args()
# random_seed = args.seed
random_seed = 42
np.random.seed(random_seed)
tf.random.set_seed(random_seed)

from sklearn.model_selection import GridSearchCV


print('-------------start!---------------------')

# === 1. read data
xls = pd.read_excel(
    '../ParleysRWIS_20211101-20230430.xlsx',
    sheet_name=['RwisAtmosphericData', 'RwisSurfaceData'],
    parse_dates=['SampleTime'],
    engine='openpyxl'
)
df1 = xls['RwisAtmosphericData']
df2 = xls['RwisSurfaceData']
df1['SampleTime'] = pd.to_datetime(df1['SampleTime'])
df2['SampleTime'] = pd.to_datetime(df2['SampleTime'])
df1['RwisControllerIntId'] = df1['RwisControllerIntId'].astype(str)
df2['RwisControllerIntId'] = df2['RwisControllerIntId'].astype(str)
df = pd.merge(df1, df2, on=['SampleTime','RwisControllerIntId'], how='inner', suffixes=('_atm','_surf'))

# === 2. delete useless feature
cols_to_drop = [
    'Pressure', 'SnowIntervalDepth','PrecipitationIntervalAmount','PrecipitationAmount',
    'RoadStatusText','iceCurCondIndex','iceAvgCondIndex','iceCurCondCode','iceAvgCondCode',
    'iceCurFricIndex','iceAvgFricIndex','iceCurFricCode','iceAvgFricCode','iceGrip','iceXValue',
    'iceYValue','iceYXRatio','iceLens','SurfaceSensorError','SurfaceSalinity','SurfaceFreezePoint','RoadStatus',
    'RoadStatus','RoadTemp','iceRoadTemp','RwisAtmoEquipmentIntId','WindSpeedGustTimestamp',
    'RwisSurfaceEquipmentIntId','WindDirection','WindSpeedAverage','WindSpeedGust','Visibility',
    'BatteryVoltage','dstInputVoltage','dstHardwareStatus','SurfaceStatusText','dscRoadStatusText','SubSurfaceTemp',
    'AirTemp_atm','DewPoint_atm','RelativeHumidity_atm','dscLevelOfGrip','dstRoadTemp','WinterRoadWeatherIndex',
    'StormIntensityIndex','PerformanceMetric','dscSurfStatus','WetBulbTemp'
]
df = df.drop(columns=cols_to_drop)

# === 3. remove outliers
filters = [
    (df['PrecipitationIntensity'] != 'fog'),
#     (df['WetBulbTemp'] >= -100),
    (df['SurfaceTemp'] < 200),
    (df['AirTemp_surf'] < 100),
    (df['DewPoint_surf'] < 80) & (df['DewPoint_surf'] > -20),
#     (df['dscSurfStatus'] < 10),
    (df['RainTotal'] <= 0.1),
    (df['SnowfallRate'] <= 3),
#     (df['WinterRoadWeatherIndex'] <= 1.6),
#     (df['StormIntensityIndex'] <= 2.2),
    (df['SurfaceWaterDepth'] <= 2),
    (df['SurfaceIceDepth'] <= 0.4),
    (df['SurfaceSnowDepth'] <= 2)
]
for f in filters:
    df = df[f]
df = df.fillna(df.mean(numeric_only=True))
df = df[(df['SurfaceGrip'] >= 0) & (df['SurfaceGrip'] <= 1)].reset_index(drop=True)

# === 4. choose time diff=10min data
df = df.sort_values(['RwisControllerIntId','SampleTime']).reset_index(drop=True)
df['time_diff'] = df.groupby('RwisControllerIntId')['SampleTime'].diff()
mask5 = ((df['time_diff']==pd.Timedelta(minutes=5)) & (df['SampleTime'].dt.minute%10==5))
df2 = df[~mask5].reset_index(drop=True)
df2 = df2.sort_values(['RwisControllerIntId','SampleTime']).reset_index(drop=True)
df2['time_diff'] = df2.groupby('RwisControllerIntId')['SampleTime'].diff()
df10 = df2[df2['time_diff']==pd.Timedelta(minutes=10)].reset_index(drop=True)

# === 5. One-hot, minmax
cat_cols = ['PrecipitationIntensity','PrecipitationType']
df_enc = pd.get_dummies(df10, columns=cat_cols)
label_col = 'SurfaceGrip'
feature_cols = [c for c in df_enc.columns if c not in ['SampleTime','RwisControllerIntId','time_diff',label_col]]
scaler = MinMaxScaler()
df_enc[feature_cols] = scaler.fit_transform(df_enc[feature_cols])

print('-------------finish data preprocessing!---------------------')

-------------start!---------------------
-------------finish data preprocessing!---------------------


In [5]:
def build_rnn_dataset(df, feature_cols, label_col='SurfaceGrip', window=10):
    X_seq, y_seq, groups = [], [], []
    for gid, grp in df.groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp)):
            # 历史 window 条特征
            hist_feats = grp.iloc[i-window:i][feature_cols].values    # (window, n_features)
            
            # 当前时刻特征
            curr_feat  = grp.iloc[i][feature_cols].values.reshape(1, -1)  # (1, n_features)
            
            # 拼成 (window+1, n_features)
            seq = np.vstack([hist_feats, curr_feat])
            
            X_seq.append(seq)
            y_seq.append(grp.iloc[i][label_col])  # 目标仍然是当前时刻的真实 SurfaceGrip
            groups.append(gid)
    
    return np.array(X_seq), np.array(y_seq), np.array(groups)



def build_tabular_dataset(df, feature_cols, label_col='SurfaceGrip', window=10):
    X_tab, y_tab, groups = [], [], []
    for gid, grp in df.sort_values(['RwisControllerIntId','SampleTime']).groupby('RwisControllerIntId'):
        grp = grp.reset_index(drop=True)
        for i in range(window, len(grp)):
            tgt = grp.iloc[i]
            X_tab.append(tgt[feature_cols].values)
            y_tab.append(tgt[label_col]); groups.append(gid)
    return np.array(X_tab), np.array(y_tab), np.array(groups)

X_seq, y_seq, groups_seq = build_rnn_dataset(df_enc, feature_cols, label_col)
X_tab, y_tab, groups_tab = build_tabular_dataset(df_enc, feature_cols, label_col)
assert np.array_equal(groups_seq, groups_tab)

# === 7. data balanced
def balance_y_distribution(X_seq, X_tab, y, bins=50, max_per_bin=1000, random_state=random_seed):
    np.random.seed(random_state)
    edges = np.linspace(np.min(y), np.max(y), bins+1)
    idx = np.digitize(y, bins=edges, right=True)
    sel = []
    for b in range(1, bins+1):
        ids = np.where(idx==b)[0]
        if len(ids)>max_per_bin:
            ids = np.random.choice(ids, max_per_bin, replace=False)
        sel.extend(ids)
    sel = np.array(sel)
    return X_seq[sel], X_tab[sel], y[sel]

# === 8. LOOCV ===
def evaluate(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mse   = mean_squared_error(y_true, y_pred)
    rmse  = np.sqrt(mse)
    mae   = mean_absolute_error(y_true, y_pred)
    medae = median_absolute_error(y_true, y_pred)
    mape  = mean_absolute_percentage_error(y_true, y_pred)

    y_pred_pos = np.maximum(y_pred, 0)
    msle  = mean_squared_log_error(y_true, y_pred_pos)
    rmsle = np.sqrt(msle)

    smape = np.mean(
        np.abs(y_true - y_pred) /
        ((np.abs(y_true) + np.abs(y_pred)) / 2 + 1e-8)
    )

    if len(y_true) > 1:
        naive_mae = np.mean(np.abs(np.diff(y_true)))
        mase = mae / (naive_mae + 1e-8)
    else:
        mase = np.nan

    ev    = explained_variance_score(y_true, y_pred)
    mx    = max_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)

    return {
        'MSE':      mse,
        'RMSE':     rmse,
        'MAE':      mae,
        'MedAE':    medae,
        'MAPE':     mape,
        'sMAPE':    smape,
        'MSLE':     msle,
        'RMSLE':    rmsle,
        'MASE':     mase,
        'ExplVar':  ev,
        'MaxError': mx,
        'R2':       r2
    }
    
    


def train_svr(X, y):
    space = {
        'C':       Real(1e-3, 1e3, prior='log-uniform'),
        'gamma':   Real(1e-4, 1e-1, prior='log-uniform'),
        'epsilon': Real(1e-3, 1.0, prior='log-uniform')
    }
    opt = BayesSearchCV(
        SVR(kernel='rbf'),
        space,
        n_iter=30,
        scoring='neg_mean_squared_error',
        cv=3,
        random_state=random_seed,
        n_jobs=-1
    )
    opt.fit(X, y)
    return opt.best_estimator_


In [6]:
from sklearn.model_selection import train_test_split

# --- 确保 init_model_excel 包含 dataset 列 ---
def init_model_excel(model_name):
    path = f'{model_name}_results.xlsx'
    if not os.path.exists(path):
        wb = Workbook()
        ws = wb.active
        ws.title = 'Metrics'
        # 表头：所有指标 + fold, station, dataset
        headers = list(evaluate(np.zeros(1), np.zeros(1)).keys()) + ['fold','station','dataset']
        ws.append(headers)
        wb.save(path)
    return path

def append_model_metrics(model_name, row_dict):
    path = f'{model_name}_results.xlsx'
    wb = load_workbook(path)
    ws = wb['Metrics']
    # 读表头
    header = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
    # 按表头顺序排值
    row = [row_dict.get(col, "") for col in header]
    ws.append(row)
    wb.save(path)
    return path

def train_dt(X, y):
    space = {
        # 允许树深从 2 到 50
        'max_depth':         Integer(2, 50),
        # 每个内部节点至少需要 2 到 100 个样本才能split
        'min_samples_split': Integer(2, 100),
        # 叶子节点至少保留 1 到 50 个样本
        'min_samples_leaf':  Integer(1, 50),
    }
    opt = BayesSearchCV(
        DecisionTreeRegressor(random_state=random_seed),
        space,
        n_iter=30,                           # 迭代次数保持 30
        scoring='neg_mean_squared_error',
        cv=3,
        random_state=random_seed,
        n_jobs=-1
    )
    opt.fit(X, y)
    return opt.best_estimator_

def train_xgb(X, y):
    space = {
        # 树的棵数可以从几十到上千
        'n_estimators':     Integer(50, 500),
        # 树的深度一般不超过 10–15
        'max_depth':        Integer(2, 15),
        # 学习率从 1e-4 到 0.3
        'learning_rate':    Real(1e-4, 3e-1, prior='log-uniform'),
        # 子采样比例
        'subsample':        Real(0.5, 1.0),
        # 每棵树列采样比例
        'colsample_bytree': Real(0.5, 1.0),
        # 最小分裂损失权重
        'gamma':            Real(0.0, 5.0),
        # L1 / L2 正则
        'reg_alpha':        Real(1e-5, 1e+1, prior='log-uniform'),
        'reg_lambda':       Real(1e-5, 1e+1, prior='log-uniform'),
        # 可选：控制叶子节点最小权重和最小样本数，帮助防止过拟合
        'min_child_weight': Integer(1, 20),
        'max_delta_step':   Integer(0, 10)
    }

    opt = BayesSearchCV(
        XGBRegressor(objective='reg:squarederror', random_state=random_seed),
        space,
        n_iter=30,                # 或根据时间预算改成 20/50
        scoring='neg_mean_squared_error',
        cv=3,
        random_state=random_seed,
        n_jobs=-1
    )
    opt.fit(X, y)
    return opt.best_estimator_

def train_rf(X, y):
    space = {
        'n_estimators':       Integer(50, 500),
        'max_depth':          Integer(2, 50),
        'min_samples_split':  Integer(2, 100),
        'min_samples_leaf':   Integer(1, 50),
        # 只试两种内建策略，去掉 'auto'
        'max_features':       Categorical(['sqrt','log2'])
        # — 或者，你想试比例，就：
        # 'max_features':    Real(0.1, 1.0)
    }
    opt = BayesSearchCV(
        RandomForestRegressor(random_state=random_seed),
        space,
        n_iter=30,
        scoring='neg_mean_squared_error',
        cv=3,
        random_state=random_seed,
        n_jobs=-1
    )
    opt.fit(X, y)
    return opt.best_estimator_

def train_mlp(X, y):
    base = MLPRegressor(
        hidden_layer_sizes=(100, 50),
        solver='adam',
        max_iter=300,
        random_state=random_seed
    )
    space = {
        'activation': Categorical(['relu', 'tanh']),
        'alpha':      Real(1e-5, 1e-1, prior='log-uniform')
    }
    opt = BayesSearchCV(
        base,
        space,
        n_iter=20,
        scoring='neg_mean_squared_error',
        cv=3,
        random_state=random_seed,
        n_jobs=-1
    )
    opt.fit(X, y)
    return opt.best_estimator_

def train_rnn(X, y):
    def build_model(learning_rate=1e-3):
        m = Sequential([
            SimpleRNN(64, input_shape=X.shape[1:]),
            Dense(1)
        ])
        m.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)]
    )

    param_grid = {
        'epochs': [200, 300, 400]
    }
    grid = GridSearchCV(
        keras_reg,
        param_grid,
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )
    # 搜索并 refit 最优模型
    grid.fit(X, y)
    best = grid.best_estimator_
    # 拿到训练过程的 History 对象
    hist = best.history_
    return best, hist


# ——— LSTM ———
def train_lstm(X, y):
    def build_model(learning_rate=1e-3):
        m = Sequential([
            LSTM(64, input_shape=X.shape[1:]),
            Dense(1)
        ])
        m.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)]
    )

    param_grid = {
        'epochs': [200, 300, 400]
    }
    grid = GridSearchCV(
        keras_reg,
        param_grid,
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )
    # 搜索并 refit 最优模型
    grid.fit(X, y)
    best = grid.best_estimator_
    # 拿到训练过程的 History 对象
    hist = best.history_
    return best, hist

def train_gru(X, y):
    def build_model(learning_rate=1e-3):
        m = Sequential([
            GRU(64, input_shape=X.shape[1:]),
            Dense(1)
        ])
        m.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )

    param_grid = {
        'epochs': [200, 300, 400]
    }
    grid = GridSearchCV(
        keras_reg,
        param_grid,
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )
    # 1) 搜索 & refit
    grid.fit(X, y)
    best = grid.best_estimator_
    # 2) 拿到训练过程 history
    hist = best.history_
    # 3) 返回模型和 history
    return best, hist

from tensorflow.keras.layers import Input, GRU, Dense, Softmax, Multiply, Lambda
from tensorflow.keras.models import Model
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV

def train_gru_attention(X, y):
    timesteps, features = X.shape[1], X.shape[2]

    def build_model(learning_rate=1e-3):
        inp = Input(shape=(timesteps, features))
        # 1) encode sequence
        x = GRU(64, return_sequences=True)(inp)  # shape=(batch, T, 64)
        # 2) compute attention scores
        #    score_t = tanh(W·h_t + b) -> scalar per time
        e = Dense(1, activation='tanh')(x)       # shape=(batch, T, 1)
        e = Lambda(lambda z: tf.squeeze(z, axis=-1))(e)  # shape=(batch, T)
        α = Softmax(axis=1)(e)                   # shape=(batch, T)
        α = Lambda(lambda z: tf.expand_dims(z, -1))(α)  # shape=(batch, T, 1)
        # 3) context = sum_t α_t · h_t
        context = Multiply()([x, α])             # shape=(batch, T, 64)
        context = Lambda(lambda z: tf.reduce_sum(z, axis=1))(context)  # (batch,64)
        # 4) regression head
        out = Dense(1)(context)

        m = Model(inputs=inp, outputs=out)
        m.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )

    param_grid = {
        'epochs': [200, 300, 400,500]
    }
    grid = GridSearchCV(
        keras_reg,
        param_grid,
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )
    grid.fit(X, y)

    best = grid.best_estimator_
    hist = best.history_
    return best, hist


# ——— Stacked LSTM (only epoch grid search) ———
def train_stacked_lstm(X, y):
    # 1) define a two-layer LSTM factory with fixed units and learning rate
    def build_model(learning_rate=1e-3):
        m = Sequential([
            LSTM(64, return_sequences=True, input_shape=X.shape[1:]),
            LSTM(32),
            Dense(1)
        ])
        m.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        return m

    # 2) wrap with scikeras, fix batch_size=32 and add early stopping
    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)]
    )

    # 3) only search over epochs
    param_grid = {
        'epochs': [200, 300, 400]
    }

    grid = GridSearchCV(
        keras_reg,
        param_grid,
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )

    # 4) fit & pull out best estimator + its History
    grid.fit(X, y)
    best = grid.best_estimator_
    hist = best.history_

    return best, hist

# new model --------------------------------

from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, Activation, Add, GlobalAveragePooling1D, Dense
from tensorflow.keras.models import Model
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV
from tensorflow.keras.callbacks import EarlyStopping

def train_tcn(X, y):
    timesteps, features = X.shape[1], X.shape[2]

    def build_model(learning_rate=1e-3, num_filters=64, kernel_size=3):
        inp = Input(shape=(timesteps, features))
        x = inp

        # build 3 dilated residual blocks with rates 1,2,4
        for dilation_rate in [1, 2, 4]:
            # 1) causal conv
            conv = Conv1D(
                filters=num_filters,
                kernel_size=kernel_size,
                dilation_rate=dilation_rate,
                padding='causal'
            )(x)
            conv = BatchNormalization()(conv)
            conv = Activation('relu')(conv)

            # 2) 1×1 conv to match dimensions for the residual add
            res = Conv1D(num_filters, kernel_size=1, padding='same')(x)
            x = Add()([res, conv])

        # global pooling + dense head
        x = GlobalAveragePooling1D()(x)
        out = Dense(1)(x)

        m = Model(inp, out)
        m.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='mse'
        )
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        verbose=0,
        batch_size=32,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )

    # only grid-search epochs here
    param_grid = {
        'epochs': [200, 300, 400]
    }
    grid = GridSearchCV(
        keras_reg,
        param_grid,
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )
    grid.fit(X, y)

    best = grid.best_estimator_
    hist = best.history_
    return best, hist

from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    Conv1D, GlobalAveragePooling1D, Dense,
    LSTM, GRU, SimpleRNN, Dropout, Add, Activation
)
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV

# 1. Seq2Point CNN
def train_seq2point_cnn(X, y):
    timesteps, features = X.shape[1], X.shape[2]
    def build_model():
        inp = Input((timesteps, features))
        x = Conv1D(32, 3, padding='causal', activation='relu')(inp)
        x = Conv1D(32, 3, padding='causal', activation='relu')(x)
        x = GlobalAveragePooling1D()(x)
        out = Dense(1)(x)
        m = Model(inp, out)
        m.compile(optimizer='adam', loss='mse')
        return m

    keras_reg = KerasRegressor(
        model=build_model, batch_size=32, verbose=0,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )
    grid = GridSearchCV(
        keras_reg, {'epochs': [50,100,200]},
        scoring='neg_mean_squared_error', cv=3, n_jobs=1
    )
    grid.fit(X, y)
    best = grid.best_estimator_
    return best, best.history_


# 2. RNN with recurrent dropout
def train_rnn_dropout(X, y):
    timesteps, features = X.shape[1], X.shape[2]
    def build_model():
        inp = Input((timesteps, features))
        # here we use GRU with dropout on inputs and recurrent state
        x = GRU(64,
                activation='tanh',
                dropout=0.2,
                recurrent_dropout=0.2)(inp)
        out = Dense(1)(x)
        m = Model(inp, out)
        m.compile(optimizer='adam', loss='mse')
        return m

    keras_reg = KerasRegressor(
        model=build_model, batch_size=32, verbose=0,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )
    grid = GridSearchCV(
        keras_reg, {'epochs': [50,100,200]},
        scoring='neg_mean_squared_error', cv=3, n_jobs=1
    )
    grid.fit(X, y)
    best = grid.best_estimator_
    return best, best.history_


# 3. Highway / Gated Residual RNN
def train_highway_rnn(X, y):
    timesteps, features = X.shape[1], X.shape[2]
    def build_model():
        inp = Input((timesteps, features))
        # Layer 1
        h1 = SimpleRNN(64, return_sequences=True)(inp)
        # Layer 2
        h2 = SimpleRNN(64, return_sequences=True)(h1)
        # Highway gate: learn gating between h1 and h2
        gate = Dense(64, activation='sigmoid')(h2)
        # combine
        x = Add()([ gate * h2, (1-gate) * h1 ])
        # final collapse
        x = SimpleRNN(64)(x)
        out = Dense(1)(x)
        m = Model(inp, out)
        m.compile(optimizer='adam', loss='mse')
        return m

    keras_reg = KerasRegressor(
        model=build_model, batch_size=32, verbose=0,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )
    grid = GridSearchCV(
        keras_reg, {'epochs': [50,100,200]},
        scoring='neg_mean_squared_error', cv=3, n_jobs=1
    )
    grid.fit(X, y)
    best = grid.best_estimator_
    return best, best.history_


# 4. Convolutional Front-End + LSTM
def train_conv_frontend_lstm(X, y):
    timesteps, features = X.shape[1], X.shape[2]
    def build_model():
        inp = Input((timesteps, features))
        # conv front-end
        x = Conv1D(32, 3, padding='causal', activation='relu')(inp)
        x = Conv1D(32, 3, padding='causal', activation='relu')(x)
        # LSTM on extracted features
        x = LSTM(64)(x)
        out = Dense(1)(x)
        m = Model(inp, out)
        m.compile(optimizer='adam', loss='mse')
        return m

    keras_reg = KerasRegressor(
        model=build_model, batch_size=32, verbose=0,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )
    grid = GridSearchCV(
        keras_reg, {'epochs': [50,100,200]},
        scoring='neg_mean_squared_error', cv=3, n_jobs=1
    )
    grid.fit(X, y)
    best = grid.best_estimator_
    return best, best.history_

def train_stacked_lstm_dropout(X, y):
    timesteps, features = X.shape[1], X.shape[2]

    def build_model():
        inp = Input((timesteps, features))
        # first LSTM layer: return sequences for stacking
        x = LSTM(
            64,
            activation='tanh',
            dropout=0.2,           # drop 20% of inputs
            recurrent_dropout=0.2, # drop 20% of recurrent state
            return_sequences=True
        )(inp)
        # second LSTM layer: final output
        x = LSTM(
            32,
            activation='tanh',
            dropout=0.2,
            recurrent_dropout=0.2
        )(x)
        out = Dense(1)(x)
        m = Model(inp, out)
        m.compile(optimizer='adam', loss='mse')
        return m

    keras_reg = KerasRegressor(
        model=build_model,
        batch_size=32,
        verbose=0,
        callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
    )

    grid = GridSearchCV(
        keras_reg,
        {'epochs': [200, 300, 400,500]},   # just tuning epochs
        scoring='neg_mean_squared_error',
        cv=3,
        n_jobs=1
    )
    grid.fit(X, y)
    best = grid.best_estimator_
    return best, best.history_

def loocv_run(X_seq, X_tab, y, groups):
    logo = LeaveOneGroupOut()

    tab_fns = [
#         ('RF',  train_rf)
#                ('XGB', train_xgb),
#                ('MLP', train_mlp),
#                ('DT',  train_dt),
#                ('SVR', train_svr)
    ]
    seq_fns = [
                ('RNN',     train_rnn),
               ('LSTM',    train_lstm),
#                ('Stacked', train_stacked_lstm),
               ('GRU',     train_gru)
#         ('GRU_attention',train_gru_attention)
#         ('tcn',train_tcn),
#         ('seq2point_cnn',train_seq2point_cnn),
# #         ('rnn_dropout',train_rnn_dropout),
#         ('highway_rnn',train_highway_rnn),
#         ('conv_frontend_lstm',train_conv_frontend_lstm),
#         ('stacked_lstm_dropout',train_stacked_lstm_dropout)
        
    ]

    for fold, (tr, te) in enumerate(logo.split(X_tab, y, groups), start=1):
        station = groups[te][0]
        X_s_tr, X_t_tr, y_tr = X_seq[tr], X_tab[tr], y[tr]
        X_s_te, X_t_te, y_te = X_seq[te], X_tab[te], y[te]

        # 平衡
        X_s_tr, X_t_tr, y_tr = balance_y_distribution(X_s_tr, X_t_tr, y_tr)

        # 再从训练集中抽 20% 做验证
        X_t_tr_sub, X_t_val, y_tr_sub, y_val = train_test_split(
            X_t_tr, y_tr, test_size=0.2, random_state=random_seed
        )
        X_s_tr_sub, X_s_val, y_tr_sub_seq, y_val_seq = train_test_split(
            X_s_tr, y_tr, test_size=0.2, random_state=random_seed
        )

        print(f"\n=== Fold {fold}: leave station {station} ===")
        print('--- Tabular models ---')

        for name, fn in tab_fns:
            print(f'-----------{name}--------------')
            # 1) 训练（用 sub-train）
            mdl = fn(X_t_tr_sub, y_tr_sub)
            # 2) 验证集评估
            y_val_pred = mdl.predict(X_t_val)
            val_mets = evaluate(y_val, y_val_pred)
            val_mets.update({'model':name,'fold':fold,'station':station,'dataset':'val'})
            init_model_excel(name)
            append_model_metrics(name, val_mets)

            # 3) 测试集评估
            y_test_pred = mdl.predict(X_t_te)
            test_mets = evaluate(y_te, y_test_pred)
            test_mets.update({'model':name,'fold':fold,'station':station,'dataset':'test'})
            append_model_metrics(name, test_mets)

            # 4) 保存模型
            os.makedirs(f'models/{name}', exist_ok=True)
            joblib.dump(mdl, f'models/{name}/{name}_fold{fold}.joblib', compress=3)

            print(f"[{name}] val→{val_mets}, test→{test_mets}")

        print('--- Sequence models ---')

        for name, fn in seq_fns:
            print(f'-----------{name}--------------')
            # 1) 训练
            mdl, history = fn(X_s_tr_sub, y_tr_sub_seq)
            # 2) 验证集评估
            y_val_pred = mdl.predict(X_s_val).ravel()
            val_mets = evaluate(y_val_seq, y_val_pred)
            val_mets.update({'model':name,'fold':fold,'station':station,'dataset':'val'})
            init_model_excel(name)
            append_model_metrics(name, val_mets)

            # 3) 测试集评估
            y_test_pred = mdl.predict(X_s_te).ravel()
            test_mets = evaluate(y_te, y_test_pred)
            test_mets.update({'model':name,'fold':fold,'station':station,'dataset':'test'})
            append_model_metrics(name, test_mets)

            # 4) 保存 loss-vs-epoch 到同文件新 Sheet
#             df_hist = pd.DataFrame(history.history)
            df_hist = pd.DataFrame(history)
            path = f'{name}_results.xlsx'
            with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                df_hist.to_excel(writer,
                                 sheet_name=f'History_Fold{fold}',
                                 index_label='epoch')

            # 5) 保存模型
            os.makedirs(f'models/{name}', exist_ok=True)
            mdl.model_.save(f'models/{name}/{name}_fold{fold}.h5')

            print(f"[{name}] val→{val_mets}, test→{test_mets}")

In [7]:
tf.autograph.set_verbosity(0)
os.makedirs('models', exist_ok=True)

print('-------------start model training!---------------------')
start_all = time.perf_counter()

# 不需要接返回值了，因为 loocv_run 已经负责写指标和存模型
loocv_run(X_seq, X_tab, y_tab, groups_seq)

end_all = time.perf_counter()
print(f"Total runtime: {end_all-start_all:.2f}s")
print("All folds done, metrics & models have been saved incrementally.")

-------------start model training!---------------------

=== Fold 1: leave station 158 ===
--- Tabular models ---
--- Sequence models ---
-----------RNN--------------
[RNN] val→{'MSE': 0.0017848129151901125, 'RMSE': 0.042247046230359256, 'MAE': 0.027954753737585384, 'MedAE': 0.017794929444789886, 'MAPE': 0.05949268735664723, 'sMAPE': 0.05676830651010576, 'MSLE': 0.0008015162616630513, 'RMSLE': 0.02831106253150968, 'MASE': 0.16252353013746004, 'ExplVar': 0.9408981651698084, 'MaxError': 0.33884911239147186, 'R2': 0.9404527300611193, 'model': 'RNN', 'fold': 1, 'station': '158', 'dataset': 'val'}, test→{'MSE': 0.0026950231825617916, 'RMSE': 0.05191361269033192, 'MAE': 0.03687456537157334, 'MedAE': 0.02058708667755127, 'MAPE': 0.04877786808615773, 'sMAPE': 0.04993107167431642, 'MSLE': 0.0009006865631276801, 'RMSLE': 0.03001144053736308, 'MASE': 6.606565772986676, 'ExplVar': 0.649473513776224, 'MaxError': 0.46672552824020386, 'R2': 0.45663090228804626, 'model': 'RNN', 'fold': 1, 'station': '


=== Fold 4: leave station 29 ===
--- Tabular models ---
--- Sequence models ---
-----------RNN--------------
[RNN] val→{'MSE': 0.0019649556393596295, 'RMSE': 0.04432782015122816, 'MAE': 0.030590276433403715, 'MedAE': 0.020581760724383114, 'MAPE': 0.06088748059090958, 'sMAPE': 0.06128599357618385, 'MSLE': 0.0008760249001230072, 'RMSLE': 0.02959771781950438, 'MASE': 0.1966363905011777, 'ExplVar': 0.9267634344088033, 'MaxError': 0.2877832353115082, 'R2': 0.9236385773980311, 'model': 'RNN', 'fold': 4, 'station': '29', 'dataset': 'val'}, test→{'MSE': 0.3537208245070275, 'RMSE': 0.5947443354139891, 'MAE': 0.41882696338432696, 'MedAE': 0.22940075397491455, 'MAPE': 0.5154616255040732, 'sMAPE': 0.7479963343016047, 'MSLE': 0.11363966404156929, 'RMSLE': 0.337104826488096, 'MASE': 66.54282661612098, 'ExplVar': -27.070907446371407, 'MaxError': 1.7047590017318726, 'R2': -51.83466580514344, 'model': 'RNN', 'fold': 4, 'station': '29', 'dataset': 'test'}
-----------LSTM--------------
[LSTM] val→{'MSE'